# GINO Evolution With Shared FNO Latent and MLP Super-Resolution Decoder

Copy of notebook1, but the priority 1 is implemented - u and grad u forming a closed loop using the residuals technique - refer KELM paper.

In [1]:
from __future__ import annotations

RUN_TAG = "gino_evolution_sharedlatent_2"
RESULTS_SUBDIR = "pt_model/sharedlatent_GINO_2"

import inspect
import json
import math
import os
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as error:
    GNOBlock = None
    GNO_IMPORT_ERROR = error

try:
    from neuralop.models import FNO
except Exception as error:
    FNO = None
    FNO_IMPORT_ERROR = error

SEED = int(os.environ.get("EVOLUTION_SR_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def _repo_paths() -> Tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    if (cwd / "GINO_evolution").is_dir():
        return cwd, cwd
    if cwd.name == "GINO_evolution":
        return cwd.parent, cwd.parent
    if (cwd / "FINAL").is_dir():
        return cwd, cwd / "FINAL"
    if cwd.name == "FINAL":
        return cwd.parent, cwd
    return cwd, cwd


def _int_or_none(raw: str) -> Optional[int]:
    raw = str(raw).strip().lower()
    if raw in {"", "none", "null", "all", "0"}:
        return None
    return int(raw)


def _csv_env(name: str, default: Sequence[str]) -> list[str]:
    raw = os.environ.get(name, "").strip()
    return [x.strip() for x in raw.split(",") if x.strip()] if raw else list(default)

REPO_ROOT, FINAL_DIR = _repo_paths()
DATASET_CANDIDATES = [
    REPO_ROOT / "processed_data_split3" / "particle_evolution_dataset.npz",
    REPO_ROOT / "process_data_evolution" / "particle_evolution_dataset.npz",
    FINAL_DIR / "processed_data" / "particle_evolution_dataset.npz",
    FINAL_DIR / "processed_data_evolution" / "particle_evolution_dataset.npz",
    FINAL_DIR / "process_data_evolution" / "particle_evolution_dataset.npz",
]
DATASET_PATH = Path(os.environ.get("EVOLUTION_DATASET", "")).expanduser()
if str(DATASET_PATH) in {"", "."}:
    DATASET_PATH = next((candidate for candidate in DATASET_CANDIDATES if candidate.exists()), DATASET_CANDIDATES[0])
if not DATASET_PATH.is_absolute():
    DATASET_PATH = (Path.cwd() / DATASET_PATH).resolve()
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing particle_evolution_dataset.npz: {DATASET_PATH}")

RESULTS_DIR = REPO_ROOT / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = RESULTS_DIR / f"{RUN_TAG}_best_model.pt"
HISTORY_PATH = RESULTS_DIR / f"{RUN_TAG}_history.json"

DEFAULT_INPUT_CHANNELS = [
    "u_x", "u_y", "u_z",
    "sigma",
    "geom_dist",
    "Gamma_x", "Gamma_y", "Gamma_z",
    "gradU_xx", "gradU_xy", "gradU_xz",
    "gradU_yx", "gradU_yy", "gradU_yz",
    "gradU_zx", "gradU_zy", "gradU_zz",
]

DEFAULT_GLOBAL_CONDITION_CHANNELS = ["angle_of_attack", "freestream_magnitude", "phase"]
CFG = {
    "seed": SEED,

    "run_tag": RUN_TAG,

    "epochs": int(os.environ.get("EVOLUTION_SR_EPOCHS", "80")),

    "lr": float(os.environ.get("EVOLUTION_SR_LR", "3e-4")),

    "weight_decay": float(os.environ.get("EVOLUTION_SR_WEIGHT_DECAY", "3e-5")),

    "eval_every": int(os.environ.get("EVOLUTION_SR_EVAL_EVERY", "2")),

    "batch_size": 1,

    "num_workers": int(os.environ.get("EVOLUTION_SR_NUM_WORKERS", "0")),

    "maximum_input_particles": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_INPUT_PARTICLES", "4096")),

    "maximum_train_query_points": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_TRAIN_QUERY_POINTS", "2048")),

    "maximum_eval_query_points": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_EVAL_QUERY_POINTS", "12000")),

    "maximum_eval_batches": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_EVAL_BATCHES", "32")),

    "gradient_accumulation_steps": int(os.environ.get("EVOLUTION_SR_ACCUM_STEPS", "4")),

    "grad_clip_norm": float(os.environ.get("EVOLUTION_SR_GRAD_CLIP", "1.0")),

    "latent_res": int(os.environ.get("EVOLUTION_SR_LATENT_RES", "24")),

    "hidden_channels": int(os.environ.get("EVOLUTION_SR_HIDDEN", "128")),

    "fno_layers": int(os.environ.get("EVOLUTION_SR_FNO_LAYERS", "6")),

    "fno_modes": int(os.environ.get("EVOLUTION_SR_FNO_MODES", "6")),

    "gno_radius": float(os.environ.get("EVOLUTION_SR_GNO_RADIUS", "0.15")),
    "delta_gno_radius": float(os.environ.get("EVOLUTION_SR_DELTA_GNO_RADIUS", "0.45")),

    "mlp_layers": int(os.environ.get("EVOLUTION_SR_MLP_LAYERS", "3")),

    "mlp_hidden": int(os.environ.get("EVOLUTION_SR_MLP_HIDDEN", "128")),

    "query_pos_encoding_frequencies": int(os.environ.get("EVOLUTION_SR_QUERY_PE_FREQS", "4")),

    "loss_weighting": os.environ.get("EVOLUTION_SR_LOSS_WEIGHTING", "uncertainty"),

    "field_loss_weight": float(os.environ.get("EVOLUTION_SR_FIELD_WEIGHT", "1.0")),

    "rollout_steps_max": int(os.environ.get("EVOLUTION_SR_ROLLOUT_STEPS_MAX", "1")),
    "rollout_loss_weight_max": float(os.environ.get("EVOLUTION_SR_ROLLOUT_WEIGHT_MAX", "0.2")),
    "rollout_loss_weight_min": float(os.environ.get("EVOLUTION_SR_ROLLOUT_WEIGHT_MIN", "0.01")),
    "lr_warmup_epochs": int(os.environ.get("EVOLUTION_SR_WARMUP_EPOCHS", "5")),
    "lr_plateau_patience": int(os.environ.get("EVOLUTION_SR_PLATEAU_PATIENCE", "8")),
    "lr_plateau_factor": float(os.environ.get("EVOLUTION_SR_PLATEAU_FACTOR", "0.5")),

    "sr_grid_resolution": int(os.environ.get("EVOLUTION_SR_GRID_RES", "96")),
    "predict_delta_u": os.environ.get("EVOLUTION_SR_PREDICT_DELTA_U", "0").strip().lower() not in {"0", "false", "no", "off"},

    "input_channels": _csv_env("EVOLUTION_SR_INPUT_CHANNELS", DEFAULT_INPUT_CHANNELS),
    "global_condition_channels": _csv_env("EVOLUTION_SR_GLOBAL_CHANNELS", DEFAULT_GLOBAL_CONDITION_CHANNELS),
}

print("Dataset:", DATASET_PATH)
print("Results:", RESULTS_DIR)
print("Device :", DEVICE)
print(json.dumps(CFG, indent=2))


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Dataset: /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/processed_data_split3/particle_evolution_dataset.npz
Results: /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/pt_model/sharedlatent_GINO_2
Device : cuda
{
  "seed": 42,
  "run_tag": "gino_evolution_sharedlatent_2",
  "epochs": 80,
  "lr": 0.0003,
  "weight_decay": 3e-05,
  "eval_every": 2,
  "batch_size": 1,
  "num_workers": 0,
  "maximum_input_particles": 4096,
  "maximum_train_query_points": 2048,
  "maximum_eval_query_points": 12000,
  "maximum_eval_batches": 32,
  "gradient_accumulation_steps": 4,
  "grad_clip_norm": 1.0,
  "latent_res": 24,
  "hidden_channels": 128,
  "fno_layers": 6,
  "fno_modes": 6,
  "gno_radius": 0.15,
  "delta_gno_radius": 0.45,
  "mlp_layers": 3,
  "mlp_hidden": 128,
  "query_pos_encoding_frequen

In [2]:
def as_context(obj) -> Dict:
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "item"):
        item = obj.item()
        if isinstance(item, dict):
            return item
    return dict(obj)


dataset_file = np.load(DATASET_PATH, allow_pickle=True)
feature_names_all = [str(x) for x in dataset_file["feature_names"].tolist()]
target_names = [str(x) for x in dataset_file["target_names"].tolist()]
field_target_names = [str(x) for x in dataset_file["field_target_names"].tolist()] if "field_target_names" in dataset_file.files else ["u_x", "u_y", "u_z"]
expected_delta_targets = ["dx", "dy", "dz", "dGamma_x", "dGamma_y", "dGamma_z", "dsigma", "delta_u_x", "delta_u_y", "delta_u_z"]
if target_names != expected_delta_targets:
    raise RuntimeError(f"Expected 10-channel delta target {expected_delta_targets}; got {target_names}")
if len(field_target_names) < 3 or field_target_names[:3] != ["u_x", "u_y", "u_z"]:
    raise RuntimeError(f"Field targets must start with ['u_x','u_y','u_z']; got {field_target_names}")

missing_channels = [name for name in CFG["input_channels"] if name not in feature_names_all]
if missing_channels:
    raise KeyError(f"Missing input channels {missing_channels}; available={feature_names_all}")
for required_key in ["query_coords", "targets_velocity_field", "targets_velocity_field_norm", "field_query_mask"]:
    if required_key not in dataset_file.files:
        raise KeyError(f"Missing {required_key!r}. Regenerate the dataset with the updated processdata_evolution_u.py")

active_input_feature_indices = [feature_names_all.index(name) for name in CFG["input_channels"]]
coord_feature_indices = [feature_names_all.index(name) for name in ("x", "y", "z")]
state_feature_indices = [feature_names_all.index(name) for name in ("x", "y", "z", "Gamma_x", "Gamma_y", "Gamma_z", "sigma")]
velocity_feature_indices = [feature_names_all.index(name) for name in ("u_x", "u_y", "u_z")]
feature_names = [feature_names_all[i] for i in active_input_feature_indices]

delta_channels = 10 if bool(CFG["predict_delta_u"]) else 7

frame_contexts = list(dataset_file["pair_contexts"] if "pair_contexts" in dataset_file.files else dataset_file["frame_contexts"])
frame_ranges = list(dataset_file["pair_ranges"] if "pair_ranges" in dataset_file.files else dataset_file["frame_ranges"])
rollout_cases = [str(x) for x in dataset_file["rollout_cases"].tolist()] if "rollout_cases" in dataset_file.files else []
rollout_true_states = list(dataset_file["rollout_true_states"]) if "rollout_true_states" in dataset_file.files else []
rollout_phases = list(dataset_file["rollout_phases"]) if "rollout_phases" in dataset_file.files else []
case_to_nmin = {str(case): int(np.asarray(seq).shape[1]) for case, seq in zip(rollout_cases, rollout_true_states)}
pair_context_map = {int(i): as_context(frame_contexts[int(i)]) for i in range(len(frame_contexts))}
case_to_rollout_index = {case: i for i, case in enumerate(rollout_cases)}
case_to_frame_index = {}
for case_idx, case in enumerate(rollout_cases):
    seq = np.asarray(rollout_true_states[case_idx])
    if seq.ndim >= 1:
        case_to_frame_index[case] = {str(j).zfill(6): j for j in range(seq.shape[0])}

inputs_t = np.asarray(dataset_file["inputs_t"], dtype=np.float32)
targets_delta_norm_all = np.asarray(dataset_file["targets_delta_norm"], dtype=np.float32)
query_coords_all = np.asarray(dataset_file["query_coords"], dtype=np.float32)
targets_field_norm_all = np.asarray(dataset_file["targets_velocity_field_norm"], dtype=np.float32)
targets_next_state_all = np.asarray(dataset_file["targets_next_state"], dtype=np.float32) if "targets_next_state" in dataset_file.files else None
field_query_mask_all = np.asarray(dataset_file["field_query_mask"], dtype=bool)

train_pair_ids = np.asarray(dataset_file["train_pair_ids"], dtype=np.int64)
val_pair_ids = np.asarray(dataset_file["val_pair_ids"], dtype=np.int64) if "val_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)
test_pair_ids = np.asarray(dataset_file["test_pair_ids"], dtype=np.int64) if "test_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)
if len(val_pair_ids) == 0 and len(train_pair_ids) > 4:
    val_pair_ids = train_pair_ids[2::5]
    val_set = set(int(i) for i in val_pair_ids)
    train_pair_ids = np.asarray([i for i in train_pair_ids if int(i) not in val_set], dtype=np.int64)

input_mean_all = np.asarray(dataset_file["in_mean"], dtype=np.float32).reshape(-1)
input_std_all = np.maximum(np.asarray(dataset_file["in_std"], dtype=np.float32).reshape(-1), 1e-8)
input_mean = input_mean_all[active_input_feature_indices]
input_std = input_std_all[active_input_feature_indices]
target_mean = np.asarray(dataset_file["out_mean"], dtype=np.float32).reshape(-1)
target_std = np.maximum(np.asarray(dataset_file["out_std"], dtype=np.float32).reshape(-1), 1e-8)
field_mean = np.asarray(dataset_file["field_mean"], dtype=np.float32).reshape(-1)
field_std = np.maximum(np.asarray(dataset_file["field_std"], dtype=np.float32).reshape(-1), 1e-8)
if not (np.isfinite(field_mean).all() and np.isfinite(field_std).all()):
    raise RuntimeError("Field normalization stats contain NaN/Inf. Regenerate particle_evolution_dataset.npz with the updated preprocessing.")
coord_min = np.asarray(dataset_file["coord_min"], dtype=np.float32).reshape(3) if "coord_min" in dataset_file.files else np.min(inputs_t[:, coord_feature_indices], axis=0)
coord_span = np.maximum(np.asarray(dataset_file["coord_span"], dtype=np.float32).reshape(3) if "coord_span" in dataset_file.files else np.ptp(inputs_t[:, coord_feature_indices], axis=0), 1e-8)

def normalize_xyz(xyz: np.ndarray) -> np.ndarray:
    return np.clip((xyz.astype(np.float32) - coord_min[None, :]) / coord_span[None, :], 0.0, 1.0).astype(np.float32)

def normalize_geom_t(xyz: torch.Tensor) -> torch.Tensor:
    coord_min_t = torch.tensor(coord_min, dtype=torch.float32, device=xyz.device).view(1, 1, 3)
    coord_span_t = torch.tensor(coord_span, dtype=torch.float32, device=xyz.device).view(1, 1, 3)
    return torch.clamp((xyz - coord_min_t) / coord_span_t, 0.0, 1.0)

def sample_indices_random(n: int, cap: Optional[int]) -> np.ndarray:
    if cap is None or cap <= 0 or n <= int(cap):
        return np.arange(n, dtype=np.int64)
    return np.sort(np.random.choice(n, size=int(cap), replace=False)).astype(np.int64)

def context_global_params(context: Dict, phase: Optional[float] = None) -> np.ndarray:
    freestream = np.asarray(context.get("freestream", [0.0, 0.0, 0.0]), dtype=np.float32).reshape(-1)
    if freestream.size < 3:
        freestream = np.pad(freestream, (0, 3 - freestream.size))
    values = {
        "angle_of_attack": float(context.get("aoa_deg", 0.0)) / 45.0,
        "freestream_magnitude": float(np.linalg.norm(freestream)) / 10.0,
        "freestream_x": float(freestream[0]) / 10.0,
        "freestream_y": float(freestream[1]) / 10.0,
        "freestream_z": float(freestream[2]) / 10.0,
        "phase": float(context.get("phase_t", 0.0) if phase is None else phase),
    }
    return np.asarray([values[name] for name in CFG["global_condition_channels"]], dtype=np.float32)


class EvolutionMultiTaskDataset(Dataset):
    def __init__(self, pair_ids, max_input_particles, max_query_points):
        self.pair_ids = np.asarray(pair_ids, dtype=np.int64)
        self.max_input_particles = max_input_particles
        self.max_query_points = max_query_points

    def __len__(self):
        return len(self.pair_ids)

    def __getitem__(self, index):
        pair_id = int(self.pair_ids[index])
        start, end = int(frame_ranges[pair_id][3]), int(frame_ranges[pair_id][4])
        features_all = inputs_t[start:end]
        delta_norm_all = targets_delta_norm_all[start:end]
        n = min(features_all.shape[0], delta_norm_all.shape[0])
        context = pair_context_map[pair_id]
        case = str(context.get("case", "unknown"))
        nmin = int(case_to_nmin.get(case, n))
        n_use = min(n, nmin, self.max_input_particles if self.max_input_particles is not None else nmin)
        input_idx = np.arange(n_use, dtype=np.int64)
        input_features = features_all[input_idx]
        x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
        y_delta = delta_norm_all[input_idx, :delta_channels]
        state_phys = input_features[:, state_feature_indices]
        velocity_phys = input_features[:, velocity_feature_indices]
        

        valid_query_idx = np.flatnonzero(field_query_mask_all[pair_id])
        if valid_query_idx.size == 0:
            raise RuntimeError(f"pair_id={pair_id} has no field-grid query points")
        query_idx = valid_query_idx[sample_indices_random(len(valid_query_idx), self.max_query_points)]
        query_xyz = query_coords_all[pair_id, query_idx, :]
        y_field = targets_field_norm_all[pair_id, query_idx, :]
        out = {
            "input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])),
            "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)),
            "state_phys": torch.from_numpy(np.nan_to_num(state_phys).astype(np.float32)),
            "velocity_phys": torch.from_numpy(np.nan_to_num(velocity_phys).astype(np.float32)),
            "global_params": torch.from_numpy(context_global_params(context)),
            "output_queries": torch.from_numpy(normalize_xyz(query_xyz)),
            "query_xyz_raw": torch.from_numpy(query_xyz.astype(np.float32)),
            "y_delta": torch.from_numpy(np.nan_to_num(y_delta).astype(np.float32)),
            "y_field": torch.from_numpy(np.nan_to_num(y_field).astype(np.float32)),
            "pair_id": torch.tensor(pair_id, dtype=torch.long),
            "case": case,
        }
        out["dt"] = torch.tensor(float(context.get("dt", 0.0034)), dtype=torch.float32)
        return out

def collate_one(batch):
    item = batch[0]
    out = {k: (v.unsqueeze(0) if torch.is_tensor(v) and k != "pair_id" else v) for k, v in item.items()}
    out["pair_id"] = item["pair_id"].view(1)
    return out

train_ds = EvolutionMultiTaskDataset(train_pair_ids, CFG["maximum_input_particles"], CFG["maximum_train_query_points"])
val_ds = EvolutionMultiTaskDataset(val_pair_ids, CFG["maximum_input_particles"], CFG["maximum_eval_query_points"])
test_ds = EvolutionMultiTaskDataset(test_pair_ids, CFG["maximum_input_particles"], CFG["maximum_eval_query_points"])
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=CFG["num_workers"], collate_fn=collate_one)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_one)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_one)
print("Particle input features:", feature_names)
print("Global condition channels:", CFG["global_condition_channels"])
print("Delta targets used:", target_names[:delta_channels])
print("Field targets:", field_target_names)
print("Field mean/std:", field_mean.tolist(), field_std.tolist())
print("Splits:", {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)})


Particle input features: ['u_x', 'u_y', 'u_z', 'sigma', 'geom_dist', 'Gamma_x', 'Gamma_y', 'Gamma_z', 'gradU_xx', 'gradU_xy', 'gradU_xz', 'gradU_yx', 'gradU_yy', 'gradU_yz', 'gradU_zx', 'gradU_zy', 'gradU_zz']
Global condition channels: ['angle_of_attack', 'freestream_magnitude', 'phase']
Delta targets used: ['dx', 'dy', 'dz', 'dGamma_x', 'dGamma_y', 'dGamma_z', 'dsigma']
Field targets: ['u_x', 'u_y', 'u_z', 'dUx_dx', 'dUx_dy', 'dUx_dz', 'dUy_dx', 'dUy_dy', 'dUy_dz', 'dUz_dx', 'dUz_dy', 'dUz_dz']
Field mean/std: [0.16723033785820007, -7.788475340930745e-05, -0.42738160490989685, 0.0024141392204910517, 6.137909076642245e-05, 0.012183577753603458, -1.0436904631205834e-05, -0.007486074697226286, -3.411342913750559e-05, 0.001409107935614884, 3.506935536279343e-05, -0.010115650482475758] [0.4173305928707123, 0.30092889070510864, 0.6755390167236328, 0.14607034623622894, 0.11956420540809631, 0.12029552459716797, 0.038422729820013046, 0.04337112605571747, 0.08665864169597626, 0.102150939404964

In [3]:
def make_latent_queries(res: int, device: torch.device) -> torch.Tensor:
    line = torch.linspace(0.0, 1.0, int(res), dtype=torch.float32, device=device)
    xx, yy, zz = torch.meshgrid(line, line, line, indexing="ij")
    return torch.stack([xx, yy, zz], dim=-1).reshape(1, -1, 3)

LATENT_QUERIES = make_latent_queries(CFG["latent_res"], DEVICE)

def make_gnoblock(in_channels, out_channels, radius):
    if GNOBlock is None:
        raise RuntimeError("neuralop.layers.gno_block.GNOBlock is not available") from GNO_IMPORT_ERROR
    kwargs = dict(in_channels=in_channels, out_channels=out_channels, coord_dim=3, radius=float(radius), transform_type="linear", reduction="mean", pos_embedding_type="transformer", pos_embedding_channels=12, channel_mlp_layers=[out_channels, out_channels, out_channels])
    accepted = set(inspect.signature(GNOBlock.__init__).parameters)
    if "use_torch_scatter_reduce" in accepted:
        kwargs["use_torch_scatter_reduce"] = False
    if "use_open3d_neighbor_search" in accepted:
        kwargs["use_open3d_neighbor_search"] = False
    return GNOBlock(**{k: v for k, v in kwargs.items() if k in accepted})

def positional_encoding(coords, num_frequencies: int):
    if int(num_frequencies) <= 0:
        return coords
    freqs = (2.0 ** torch.arange(int(num_frequencies), device=coords.device, dtype=coords.dtype)).view(1, 1, -1)
    angles = coords.unsqueeze(-1) * freqs * math.pi
    return torch.cat([coords, torch.sin(angles).flatten(-2), torch.cos(angles).flatten(-2)], dim=-1)

class GINOSharedLatent(nn.Module):
    def __init__(self, in_channels, delta_channels, field_channels, global_channels, cfg):
        super().__init__()
        hidden = int(cfg["hidden_channels"])
        self.latent_res = int(cfg["latent_res"])
        self.query_pe_freqs = int(cfg["query_pos_encoding_frequencies"])
        self.lift = nn.Sequential(nn.Linear(in_channels, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.global_condition_mlp = nn.Sequential(nn.Linear(global_channels, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.encoder = make_gnoblock(hidden, hidden, cfg["gno_radius"])
        if FNO is None:
            raise RuntimeError("neuralop.models.FNO is not available; install/use neuralop with FNO support") from FNO_IMPORT_ERROR
        modes = min(int(cfg["fno_modes"]), max(self.latent_res // 2, 1))
        self.fno = FNO(n_modes=(modes, modes, modes), in_channels=hidden, out_channels=hidden, hidden_channels=hidden, n_layers=int(cfg["fno_layers"]), positional_embedding=None)
        self.delta_decoder_gno = make_gnoblock(hidden, hidden, cfg["delta_gno_radius"])
        self.particle_skip_proj = nn.Linear(hidden, hidden)
        self.delta_fusion = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.delta_head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, delta_channels))
        query_dim = 3 + 2 * 3 * self.query_pe_freqs
        width = int(cfg["mlp_hidden"])
        layers = []
        for layer_id in range(max(int(cfg["mlp_layers"]), 1)):
            layers += [nn.Linear((hidden + query_dim) if layer_id == 0 else width, width), nn.GELU()]
        layers.append(nn.Linear(width, field_channels))
        self.field_decoder = nn.Sequential(*layers)
        self.log_delta_var = nn.Parameter(torch.zeros(()))
        self.log_field_var = nn.Parameter(torch.zeros(()))

    def apply_gno(self, block, source_coords, query_coords, source_features):
        return block(y=source_coords, x=query_coords, f_y=source_features)

    def encode_process(self, input_geom, latent_queries, x, global_params):
        base_latent = latent_queries[0]
        r = self.latent_res
        grids, flat_latents, particle_skips = [], [], []
        for b in range(x.shape[0]):
            h_raw = self.lift(x[b])
            latent = self.apply_gno(self.encoder, source_coords=input_geom[b], query_coords=base_latent, source_features=h_raw)
            if latent.ndim == 3:
                latent = latent.squeeze(0)
            grid = latent.reshape(r, r, r, -1).permute(3, 0, 1, 2).unsqueeze(0)
            cond = self.global_condition_mlp(global_params[b]).view(1, -1, 1, 1, 1)
            processed_grid = self.fno(grid + cond)
            processed_flat = processed_grid.squeeze(0).permute(1, 2, 3, 0).reshape(-1, processed_grid.shape[1])
            grids.append(processed_grid)
            flat_latents.append(processed_flat)
            particle_skips.append(self.particle_skip_proj(h_raw))
        return base_latent, grids, flat_latents, particle_skips

    def sample_grid(self, grid, queries):
        q = queries.clamp(0.0, 1.0)
        sample_grid = (q * 2.0 - 1.0).view(1, -1, 1, 1, 3)
        sampled = torch.nn.functional.grid_sample(grid, sample_grid, align_corners=True, mode="bilinear")
        return sampled.squeeze(0).squeeze(-1).squeeze(-1).transpose(0, 1)

    def forward(self, input_geom, latent_queries, output_queries, x, global_params):
        base_latent, grids, flat_latents, particle_skips = self.encode_process(input_geom, latent_queries, x, global_params)
        delta_outputs, field_outputs = [], []
        for b, (grid, flat_latent, skip) in enumerate(zip(grids, flat_latents, particle_skips)):
            particle_latent = self.apply_gno(self.delta_decoder_gno, source_coords=base_latent, query_coords=input_geom[b], source_features=flat_latent)
            if particle_latent.ndim == 3:
                particle_latent = particle_latent.squeeze(0)
            fused = self.delta_fusion(torch.cat([particle_latent, skip], dim=-1))
            delta_outputs.append(self.delta_head(fused))
            q = output_queries[b].clamp(0.0, 1.0)
            sampled = self.sample_grid(grid, q)
            field_outputs.append(self.field_decoder(torch.cat([sampled, positional_encoding(q.unsqueeze(0), self.query_pe_freqs).squeeze(0)], dim=-1)))
        return torch.stack(delta_outputs, dim=0), torch.stack(field_outputs, dim=0)

model = GINOSharedLatent(len(feature_names), delta_channels, len(field_target_names), len(CFG["global_condition_channels"]), CFG).to(DEVICE)
target_mean_t = torch.tensor(target_mean[:delta_channels], dtype=torch.float32, device=DEVICE).view(1, 1, -1)
target_std_t = torch.tensor(target_std[:delta_channels], dtype=torch.float32, device=DEVICE).view(1, 1, -1)
field_mean_t = torch.tensor(field_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
field_std_t = torch.tensor(field_std, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
print(model)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


GINOSharedLatent(
  (lift): Sequential(
    (0): Linear(in_features=17, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (global_condition_mlp): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (encoder): GNOBlock(
    (pos_embedding): SinusoidalEmbedding()
    (neighbor_search): NeighborSearch()
    (integral_transform): IntegralTransform(
      (channel_mlp): LinearChannelMLP(
        (fcs): ModuleList(
          (0): Linear(in_features=144, out_features=128, bias=True)
          (1-2): 2 x Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (fno): FNO(
    (fno_blocks): FNOBlocks(
      (convs): ModuleList(
        (0-5): 6 x SpectralConv(
          (weight): DenseTensor(shape=torch.Size([128, 128, 6, 6, 4]), rank=None)
        )
      )
      (fno_

In [4]:
def move_batch(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

def predict(batch):
    return model(batch["input_geom"], LATENT_QUERIES, batch["output_queries"], batch["x"], batch["global_params"])

def denormalize_delta(y):
    return y * target_std_t + target_mean_t

def denormalize_field(y):
    return y * field_std_t + field_mean_t

def relative_l2(pred, target, eps=1e-8):
    pred = torch.nan_to_num(pred)
    target = torch.nan_to_num(target)
    numerator = torch.linalg.norm((pred - target).reshape(pred.shape[0], -1), dim=1)
    denominator = torch.linalg.norm(target.reshape(target.shape[0], -1), dim=1).clamp_min(eps)
    return numerator / denominator

def rollout_curriculum(epoch):
    frac = min(max(epoch / float(max(CFG["epochs"], 1)), 0.0), 1.0)
    weight = float(CFG["rollout_loss_weight_min"]) + frac * (float(CFG["rollout_loss_weight_max"]) - float(CFG["rollout_loss_weight_min"]))
    steps = 1 + int(round(frac * max(int(CFG["rollout_steps_max"]) - 1, 0)))
    return weight, steps

def _rollout_targets_for_batch(batch, steps: int):
    pair_id = int(batch["pair_id"].reshape(-1)[0].item())
    ctx = pair_context_map[pair_id]
    case = str(ctx.get("case", ""))
    frame_t = str(ctx.get("frame_t", "")).zfill(6)
    case_idx = case_to_rollout_index.get(case, None)
    frame_map = case_to_frame_index.get(case, {})
    if case_idx is None or frame_t not in frame_map:
        return None, None
    start_idx = frame_map[frame_t]
    seq = np.asarray(rollout_true_states[case_idx], dtype=np.float32)
    phases = np.asarray(rollout_phases[case_idx], dtype=np.float32) if len(rollout_phases) > case_idx else None
    max_steps = min(int(steps), max(seq.shape[0] - start_idx - 1, 0))
    if max_steps <= 0:
        return None, None
    n_batch = int(batch["state_phys"].shape[1])
    targets = seq[start_idx + 1 : start_idx + 1 + max_steps, :n_batch, :]
    phase_seq = None if phases is None else phases[start_idx + 1 : start_idx + 1 + max_steps]
    return torch.from_numpy(targets).to(DEVICE), (torch.from_numpy(phase_seq).to(DEVICE) if phase_seq is not None else None)

def _make_rollout_x(state_phys, velocity_phys, batch_x, phase_value=None):
    raw = torch.zeros((state_phys.shape[0], state_phys.shape[1], len(feature_names_all)), dtype=state_phys.dtype, device=state_phys.device)
    raw[:, :, state_feature_indices] = state_phys
    raw[:, :, velocity_feature_indices] = velocity_phys
    old_raw = batch_x * torch.tensor(input_std, dtype=torch.float32, device=state_phys.device).view(1, 1, -1) + torch.tensor(input_mean, dtype=torch.float32, device=state_phys.device).view(1, 1, -1)
    for local_i, global_i in enumerate(active_input_feature_indices):
        raw[:, :, global_i] = old_raw[:, :, local_i]
    raw[:, :, state_feature_indices] = state_phys
    raw[:, :, velocity_feature_indices] = velocity_phys
    if phase_value is not None and "phase" in feature_names_all:
        raw[:, :, feature_names_all.index("phase")] = phase_value
    active = raw[:, :, active_input_feature_indices]
    mean = torch.tensor(input_mean, dtype=torch.float32, device=state_phys.device).view(1, 1, -1)
    std = torch.tensor(input_std, dtype=torch.float32, device=state_phys.device).view(1, 1, -1)
    return torch.clamp(torch.nan_to_num((active - mean) / std), -8.0, 8.0)

def _global_params_for_phase(batch, phase_value):
    gp = batch["global_params"].clone()
    if "phase" in CFG["global_condition_channels"] and phase_value is not None:
        gp[:, CFG["global_condition_channels"].index("phase")] = phase_value.reshape(-1)
    return gp

def rollout_loss(batch, steps=3, step_weights=None):
    targets, target_phases = _rollout_targets_for_batch(batch, steps)
    if targets is None:
        return torch.zeros((), device=DEVICE)
    state_phys = torch.nan_to_num(batch["state_phys"]).clone()
    velocity_phys = torch.nan_to_num(batch["velocity_phys"]).clone()
    geom = batch["input_geom"].clone()
    x = batch["x"].clone()
    global_params = batch["global_params"].clone()
    total = torch.zeros((), device=DEVICE)
    weights = []
    for k in range(targets.shape[0]):
        step_batch = {"input_geom": geom, "output_queries": batch["output_queries"], "x": x, "global_params": global_params}
        delta_pred, _ = predict(step_batch)
        next_state = state_phys.clone()
        next_state[:, :, :7] = state_phys[:, :, :7] + delta_pred[:, :, :7]
        if delta_pred.shape[-1] >= 10:
            velocity_phys = velocity_phys + delta_pred[:, :, 7:10]
        w = 1.0 + (float(k) / max(targets.shape[0] - 1, 1)) if step_weights is None else float(step_weights[k])
        weights.append(w)
        total = total + w * F.mse_loss(next_state[:, :, :7], torch.nan_to_num(targets[k:k+1, :, :7]))
        state_phys = next_state
        geom = normalize_geom_t(state_phys[:, :, :3])
        phase_value = None if target_phases is None else target_phases[k:k+1].view(1, 1, 1)
        x = _make_rollout_x(state_phys, velocity_phys, batch["x"], phase_value)
        global_params = _global_params_for_phase(batch, phase_value.view(1, 1) if phase_value is not None else None)
    return total / max(sum(weights), 1e-8)

def multitask_loss(delta_pred, field_pred, y_delta, y_field):
    loss_delta = F.mse_loss(delta_pred, y_delta[:, :, :delta_pred.shape[-1]])
    loss_field = F.mse_loss(field_pred, y_field)
    if CFG["loss_weighting"].lower() == "uncertainty":
        total = torch.exp(-model.log_delta_var) * loss_delta + model.log_delta_var
        total = total + torch.exp(-model.log_field_var) * loss_field + model.log_field_var
    else:
        total = loss_delta + float(CFG["field_loss_weight"]) * loss_field
    return total, loss_delta.detach(), loss_field.detach()

def extract_gradient_tensor(x, feature_names):
    """x: (B, N, C) normalised features. Returns gradU of shape (B, N, 3, 3) in PHYSICAL units."""
    grad_indices = [feature_names.index(f"gradU_{i}{j}") for i in "xyz" for j in "xyz"]
    grad_norm = x[:, :, grad_indices]                                          # (B, N, 9)
    # Denormalise using the corresponding input statistics
    grad_mean = torch.tensor(input_mean[grad_indices], device=x.device).view(1,1,9)
    grad_std  = torch.tensor(input_std[grad_indices],  device=x.device).view(1,1,9)
    grad_phys = grad_norm * grad_std + grad_mean                               # (B, N, 9)
    return grad_phys.reshape(x.shape[0], x.shape[1], 3, 3)                     # (B, N, 3, 3)

@torch.no_grad()
def evaluate(loader, max_batches=None, epoch_for_rollout=None):
    model.eval()
    if len(loader.dataset) == 0:
        return {"loss": math.nan, "delta_loss": math.nan, "field_loss": math.nan, "delta_rel_l2": math.nan, "field_rel_l2": math.nan, "rollout_rel_l2": math.nan, "batches": 0}
    losses, delta_losses, field_losses, delta_rels, field_rels, rollout_rels = [], [], [], [], [], []
    _, rollout_steps = rollout_curriculum(epoch_for_rollout or CFG["epochs"])
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= int(max_batches):
            break
        batch = move_batch(batch)
        delta_pred, field_pred = predict(batch)
        loss, delta_loss, field_loss = multitask_loss(delta_pred, field_pred, batch["y_delta"], batch["y_field"])
        delta_pred_phys = denormalize_delta(delta_pred)
        y_delta_phys = denormalize_delta(batch["y_delta"][:, :, :delta_pred.shape[-1]])
        losses.append(float(loss.item()))
        delta_losses.append(float(delta_loss.item()))
        field_losses.append(float(field_loss.item()))
        delta_rels.append(float(relative_l2(delta_pred_phys, y_delta_phys).mean().item()))
        field_rels.append(float(relative_l2(field_pred, batch["y_field"]).mean().item()))
        rollout_rels.append(float(rollout_loss(batch, steps=rollout_steps).item()))
    return {"loss": float(np.mean(losses)) if losses else math.nan, "delta_loss": float(np.mean(delta_losses)) if delta_losses else math.nan, "field_loss": float(np.mean(field_losses)) if field_losses else math.nan, "delta_rel_l2": float(np.mean(delta_rels)) if delta_rels else math.nan, "field_rel_l2": float(np.mean(field_rels)) if field_rels else math.nan, "rollout_rel_l2": float(np.mean(rollout_rels)) if rollout_rels else math.nan, "batches": len(losses)}

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
warmup_epochs = max(int(CFG["lr_warmup_epochs"]), 0)
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=float(CFG["lr_plateau_factor"]), patience=int(CFG["lr_plateau_patience"]), min_lr=1e-6)
history, best = [], float("inf")

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    losses, delta_losses, field_losses, rollout_terms = [], [], [], []
    accum = max(int(CFG["gradient_accumulation_steps"]), 1)
    rollout_weight, rollout_steps = rollout_curriculum(epoch)

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)
        delta_pred, field_pred = predict(batch)

        # ---- Residual target computation ----
        u_true    = batch["velocity_phys"]
        Gamma     = batch["state_phys"][..., 3:6]
        dt        = batch["dt"].view(-1,1,1) if "dt" in batch else torch.tensor([0.0034], device=DEVICE).view(-1,1,1)
        gradU_true = extract_gradient_tensor(batch["x"], feature_names)

        physics_dx     = dt * u_true
        physics_dGamma = dt * torch.einsum('bnij,bnj->bni', gradU_true, Gamma)

        true_delta_phys = denormalize_delta(batch["y_delta"])
        residual_dx     = true_delta_phys[..., :3] - physics_dx
        residual_dGamma = true_delta_phys[..., 3:6] - physics_dGamma
        residual_dsigma = true_delta_phys[..., 6:7]
        residual_target = torch.cat([residual_dx, residual_dGamma, residual_dsigma], dim=-1)

        delta_pred_phys = denormalize_delta(delta_pred)[..., :7]

        loss_delta = F.mse_loss(delta_pred_phys, residual_target)
        loss_field = F.mse_loss(field_pred, batch["y_field"])

        if CFG["loss_weighting"].lower() == "uncertainty":
            loss = torch.exp(-model.log_delta_var) * loss_delta + model.log_delta_var
            loss = loss + torch.exp(-model.log_field_var) * loss_field + model.log_field_var
        else:
            loss = loss_delta + float(CFG["field_loss_weight"]) * loss_field

        rollout_term = rollout_loss(batch, steps=rollout_steps)
        total_loss = loss + rollout_weight * rollout_term
        (total_loss / accum).backward()

        if step % accum == 0 or step == len(train_loader):
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip_norm"])
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        losses.append(float(total_loss.item()))
        delta_losses.append(float(loss_delta.item()))
        field_losses.append(float(loss_field.item()))
        rollout_terms.append(float(rollout_term.item()))

    if warmup_epochs > 0 and epoch <= warmup_epochs:
        for param_group in optimizer.param_groups:
            param_group["lr"] = float(CFG["lr"]) * epoch / float(warmup_epochs)
    if epoch % CFG["eval_every"] == 0 or epoch == CFG["epochs"]:
        val = evaluate(val_loader, max_batches=CFG["maximum_eval_batches"], epoch_for_rollout=epoch)
        test = evaluate(test_loader, max_batches=min(8, CFG["maximum_eval_batches"] or 8), epoch_for_rollout=epoch)

        # ---- Activate multi‑step rollout when validation delta rel L2 is low enough ----
        rollout_activation_threshold = 0.2          # you can tune this value
        if (CFG["rollout_steps_max"] == 1 and 
            val.get("delta_rel_l2", math.nan) < rollout_activation_threshold):
            print(f"[rollout] val delta_rel_l2 = {val['delta_rel_l2']:.4f} < {rollout_activation_threshold} -> enabling multi‑step rollout")
            CFG["rollout_steps_max"] = 2            # start with 2 steps
            CFG["rollout_weight_max"] = 0.05        # keep the rollout weight small
            # The curriculum function will now ramp steps from 1 to 2 and weight from
            # rollout_weight_min to 0.05 over the remaining epochs.

        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "train_delta_loss": float(np.mean(delta_losses)), "train_field_loss": float(np.mean(field_losses)), "train_rollout_loss": float(np.mean(rollout_terms)), "log_delta_var": float(model.log_delta_var.detach().cpu()), "log_field_var": float(model.log_field_var.detach().cpu()), "val": val, "test": test, "lr": optimizer.param_groups[0]["lr"], "rollout_weight": float(rollout_weight), "rollout_steps": int(rollout_steps)}
        history.append(row)
        print(json.dumps(row, indent=2))
        plateau_metric = val["field_loss"] if np.isfinite(val["field_loss"]) else row["train_loss"]
        if epoch > warmup_epochs:
            plateau_scheduler.step(plateau_metric)
        score = val["field_rel_l2"] if np.isfinite(val["field_rel_l2"]) else row["train_loss"]
        if score < best:
            best = score
            torch.save({"checkpoint_tag": RUN_TAG, "saved_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"), "config": CFG, "model_state_dict": model.state_dict(), "feature_names_all": feature_names_all, "feature_names": feature_names, "target_names": target_names[:delta_channels], "field_target_names": field_target_names, "active_input_feature_indices": active_input_feature_indices, "coord_feature_indices": coord_feature_indices, "coord_min": coord_min, "coord_span": coord_span, "input_mean": input_mean, "input_std": input_std, "target_mean": target_mean[:delta_channels], "target_std": target_std[:delta_channels], "field_mean": field_mean, "field_std": field_std, "global_condition_channels": CFG["global_condition_channels"], "dataset_path": str(DATASET_PATH), "best_score": best, "history": history}, CKPT_PATH)
            print("saved", CKPT_PATH)
HISTORY_PATH.write_text(json.dumps(history, indent=2))


{
  "epoch": 2,
  "train_loss": 0.8497359470210292,
  "train_delta_loss": 0.009184390753851156,
  "train_field_loss": 0.8757578810719265,
  "train_rollout_loss": 0.029457543096743227,
  "log_delta_var": -0.03853769227862358,
  "log_field_var": -0.006564312148839235,
  "val": {
    "loss": 0.4261738872155547,
    "delta_loss": 0.13886108808219433,
    "field_loss": 0.32481973338872194,
    "delta_rel_l2": 0.41065812297165394,
    "field_rel_l2": 0.9360944647341967,
    "rollout_rel_l2": 0.0011019580088031944,
    "batches": 32
  },
  "test": {
    "loss": 3.888333026319742,
    "delta_loss": 3.482579949311912,
    "field_loss": 0.31197061855345964,
    "delta_rel_l2": 0.6038053780794144,
    "field_rel_l2": 0.9254883080720901,
    "rollout_rel_l2": 0.17061183498299215,
    "batches": 8
  },
  "lr": 0.00011999999999999999,
  "rollout_weight": 0.014750000000000001,
  "rollout_steps": 1
}
saved /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/pt_model/sharedlatent_GINO_2

KeyboardInterrupt: 

In [ ]:
@torch.no_grad()
def predict_u_on_field_grid(pair_id: int, grid_resolution: Optional[int] = None, y_value: Optional[float] = None, max_input_particles: Optional[int] = None):
    """Query the MLP field decoder on an x-z plane across the full coordinate extent."""
    model.eval()
    grid_resolution = int(grid_resolution or CFG["sr_grid_resolution"])
    pair_range = frame_ranges[int(pair_id)]
    start, end = int(pair_range[3]), int(pair_range[4])
    features_all = inputs_t[start:end]
    context = pair_context_map[int(pair_id)]
    case = str(context.get("case", "unknown"))
    n_use = min(features_all.shape[0], int(case_to_nmin.get(case, features_all.shape[0])))
    if max_input_particles is not None:
        n_use = min(n_use, int(max_input_particles))
    input_idx = np.arange(n_use, dtype=np.int64)
    input_features = features_all[input_idx]
    x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
    x_line = np.linspace(coord_min[0], coord_min[0] + coord_span[0], grid_resolution, dtype=np.float32)
    z_line = np.linspace(coord_min[2], coord_min[2] + coord_span[2], grid_resolution, dtype=np.float32)
    xx, zz = np.meshgrid(x_line, z_line, indexing="xy")
    if y_value is None:
        y_value = float(np.median(features_all[:, coord_feature_indices[1]]))
    yy = np.full_like(xx, float(y_value), dtype=np.float32)
    query_xyz = np.stack([xx, yy, zz], axis=-1).reshape(-1, 3)
    batch = {
        "input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])).unsqueeze(0).to(DEVICE),
        "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)).unsqueeze(0).to(DEVICE),
        "global_params": torch.from_numpy(context_global_params(context)).unsqueeze(0).to(DEVICE),
        "output_queries": torch.from_numpy(normalize_xyz(query_xyz)).unsqueeze(0).to(DEVICE),
    }
    _, field_pred_norm = predict(batch)
    u = denormalize_field(field_pred_norm).squeeze(0).cpu().numpy()
    return query_xyz, u, u.reshape(grid_resolution, grid_resolution, 3)

if len(test_ds) > 0:
    pair_id = int(test_ds.pair_ids[0])
elif len(val_ds) > 0:
    pair_id = int(val_ds.pair_ids[0])
else:
    pair_id = int(train_ds.pair_ids[0])
query_xyz, u_flat, u_grid = predict_u_on_field_grid(pair_id, grid_resolution=min(CFG["sr_grid_resolution"], 96))
u_mag = np.linalg.norm(u_grid, axis=-1)
fig, ax = plt.subplots(figsize=(7.2, 5.5), constrained_layout=True)
im = ax.imshow(u_mag, origin="lower", extent=[query_xyz[:,0].min(), query_xyz[:,0].max(), query_xyz[:,2].min(), query_xyz[:,2].max()], aspect="auto")
ax.set_xlabel("x")
ax.set_ylabel("z")
ax.set_title(f"MLP field decoder query |u|, pair_id={pair_id}")
plt.colorbar(im, ax=ax, label="|u|")
plot_path = RESULTS_DIR / f"{RUN_TAG}_sr_u_field.png"
fig.savefig(plot_path, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)
